In [6]:

# Standard Libraries

import os
import gc
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# Data Manipulation

import numpy as np
import pandas as pd


# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("Set2")


# Machine Learning

from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Miscellaneous

from tqdm import tqdm


# Data Cleaning & Preprocessing Strategy

## Objectives

In this section we will:

- Remove useless features
- Optimize memory usage
- Handle missing values
- Preserve informative missingness
- Encode categorical variables
- Engineer new features
- Prepare the dataset for both
  * Machine Learning
  * Neural Networks

Our preprocessing strategy follows practices commonly used for IEEE-CIS fraud detection: removing extremely sparse or zero-variance features, preserving informative missingness, using median imputation for numerical variables, explicit missing categories for categorical variables, and delaying scaling until after the train/validation split to avoid data leakage.





# Overall Preprocessing Pipeline

Instead of writing preprocessing randomly throughout the notebook, let's define our roadmap first.
## Data Processing Pipeline

```mermaid
flowchart LR
    A[Raw Data] --> B[Data Cleaning]
    B --> C[Feature Engineering]
    C --> D[Data Preprocessing]
    D --> E[Train / Validation Split]
    E --> F[Machine Learning Model]
```


---


# 3.1 Create a Working Copy

Never preprocess the original dataframe.

In [4]:
DATA_DIR =  Path("data/raw")

df = pd.read_csv(DATA_DIR/"features_df.csv")

In [8]:

# Working Copy
data = df

print(data.shape)

(590540, 434)


# 3.2 Remove Duplicate Rows

Although IEEE-CIS rarely contains duplicate transactions, checking is still good engineering practice.

In [9]:
duplicates = data.duplicated().sum()

print(f"Duplicate Rows : {duplicates:,}")

if duplicates > 0:

    data = data.drop_duplicates()

print(data.shape)

Duplicate Rows : 0
(590540, 434)


# 3.3 Remove Constant Features

Constant features contribute no predictive information.

In [10]:
constant_cols = [
    col
    for col in data.columns
    if data[col].nunique(dropna=False) == 1
]

print(f"Constant Columns : {len(constant_cols)}")

Constant Columns : 0


In [11]:
data.drop(columns=constant_cols,
          inplace=True)

print(data.shape)

(590540, 434)


---

# 3.4 Remove Extremely Sparse Features

This is our first real design decision.

Many IEEE-CIS features have **95–99% missing values**.

Keeping them increases memory usage and often adds noise.

A commonly used threshold is:

> Remove columns with more than **95% missing values**.

We'll make this configurable.

In [12]:

# Missing Threshold


MISSING_THRESHOLD = 95

In [13]:
missing_percent = (
    data
    .isnull()
    .mean()
    *100
)

In [14]:
high_missing_cols = (
    missing_percent[
        missing_percent > MISSING_THRESHOLD
    ]
    .index
    .tolist()
)

print(f"Columns to Remove : {len(high_missing_cols)}")

Columns to Remove : 9


In [15]:
data.drop(
    columns=high_missing_cols,
    inplace=True
)

print(data.shape)

(590540, 425)


This threshold substantially reduces dimensionality while retaining most of the predictive signal in IEEE-CIS-style pipelines.




# 3.5 Memory Optimization

Large fraud datasets consume significant RAM.

We'll reduce memory usage by downcasting numeric types.

In [16]:

# Memory Optimization


def reduce_memory_usage(df):

    start_mem = (
        df.memory_usage(deep=True)
        .sum()/1024**2
    )

    for col in df.columns:

        col_type = df[col].dtype

        if str(col_type).startswith("int"):

            df[col] = pd.to_numeric(
                df[col],
                downcast="integer"
            )

        elif str(col_type).startswith("float"):

            df[col] = pd.to_numeric(
                df[col],
                downcast="float"
            )

    end_mem = (
        df.memory_usage(deep=True)
        .sum()/1024**2
    )

    print(f"Memory Before : {start_mem:.2f} MB")
    print(f"Memory After  : {end_mem:.2f} MB")
    print(
        f"Reduction     : {100*(start_mem-end_mem)/start_mem:.2f}%"
    )

    return df

In [17]:
data = reduce_memory_usage(data)

Memory Before : 2446.11 MB
Memory After  : 1551.21 MB
Reduction     : 36.58%



# 3.6 Separate Feature Types

In [18]:
TARGET = "isFraud"

TARGET_ID = "TransactionID"

In [19]:
numerical_columns = (
    data
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
    .tolist()
)

categorical_columns = (
    data
    .select_dtypes(
        include="object"
    )
    .columns
    .tolist()
)

Remove target from feature list.

In [20]:
numerical_columns.remove(TARGET)

if TARGET_ID in numerical_columns:
    numerical_columns.remove(TARGET_ID)

In [21]:
print(f"Numerical Features : {len(numerical_columns)}")

print(f"Categorical Features : {len(categorical_columns)}")

Numerical Features : 394
Categorical Features : 29


# 3.7 Missing Value Summary

Instead of immediately imputing values, we will classify the remaining columns by missingness. This will help us decide where to add missing-value indicators and where simple imputation is sufficient.

In [22]:
missing_summary = (
    data
    .isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_summary = (
    missing_summary[
        missing_summary > 0
    ]
)

display(
    missing_summary
    .head(25)
)

dist2    93.628374
D7       93.409930
id_18    92.360721
D13      89.509263
D14      89.469469
D12      89.041047
id_04    88.768923
id_03    88.768923
D6       87.606767
id_33    87.589494
D8       87.312290
id_09    87.312290
D9       87.312290
id_10    87.312290
id_30    86.865411
id_32    86.861855
id_34    86.824771
id_14    86.445626
V146     86.123717
V149     86.123717
V156     86.123717
V157     86.123717
V155     86.123717
V154     86.123717
V161     86.123717
dtype: float64

# Why we stop here

Many notebooks immediately perform:

* Median imputation
* Label encoding
* Scaling

That is **incorrect** because we have not yet split the data.

If you compute medians, category mappings, or scaling parameters using the entire dataset, you introduce **data leakage**—information from the validation set influences the training process. The correct approach is to **split first**, then fit imputers, encoders, and scalers only on the training data before applying them to validation and test sets. This principle is fundamental to building reliable ML pipelines.
- A mistake can be often seen, is:
```py
train_test_split(...)
...
fit(RandomForest)
```

In [25]:
data.to_csv(DATA_DIR / 'cleaned_df.csv', index=False)
print("Cleaned data save in cleaned_df.csv")

Cleaned data save in cleaned_df.csv


In [26]:
import gc

def reset_memory():
    # Delete all variables.
    for var in gc.get_objects():
        try:
            del var
        except:
            pass

    # Run the garbage collector.
    gc.collect()

    print("Memory has been reset")


In [28]:
reset_memory()

Memory has been reset


In [29]:
del data